# Inundation Package - Quick Examples

Let's explore the inundation package and see what we can do with Sacramento River and Yolo Bypass data.

## Getting started

First, let's import what we need:

In [ ]:
from inundation import get_fre, get_dayflow, calc_inundation, show_cache, clear_cache
import pandas as pd
import matplotlib.pyplot as plt

## Downloading Fremont Weir data

Let's start by getting the Sacramento River stage height data from Fremont Weir:

In [ ]:
# Download Fremont Weir stage height (this might take a minute or two)
fre = get_fre()
print(f"Got {len(fre)} records")
fre.head()

What does the data look like?

In [ ]:
fre.info()

In [ ]:
# Date range
print(f"From: {fre['datetime'].min()}")
print(f"To: {fre['datetime'].max()}")
print(f"\nBasic stats on stage height (feet):")
print(fre['value'].describe())

## Getting Dayflow data

Now let's get the daily flow data for Sacramento River and Yolo Bypass:

In [ ]:
# Download Dayflow data
dayflow = get_dayflow()
print(f"Got {len(dayflow)} records")
dayflow.head(10)

In [ ]:
# What's in it?
dayflow.info()

In [ ]:
# Date range and basic stats
print(f"From: {dayflow['date'].min()}")
print(f"To: {dayflow['date'].max()}")
print(f"\nSacramento River flow (cfs):")
print(dayflow['sac'].describe())
print(f"\nYolo Bypass flow (cfs):")
print(dayflow['yolo'].describe())

## Calculating inundation

Now the main event - let's calculate when the Yolo Bypass was inundated:

In [ ]:
# Calculate inundation
inun = calc_inundation()
print(f"Got {len(inun)} days of data")
inun.head(10)

In [ ]:
# What columns do we have?
inun.info()

## Quick stats

In [ ]:
# How many days was the Bypass inundated?
total_inun_days = inun['inundation'].sum()
pct_inundated = 100 * inun['inundation'].mean()

print(f"Total inundation days: {int(total_inun_days)}")
print(f"Percentage of time inundated: {pct_inundated:.2f}%")
print(f"Max consecutive inundation: {inun['inund_days'].max()} days")

In [ ]:
# Water level stats
print(f"Sacramento River stage height (feet):")
print(f"  Mean: {inun['height_sac'].mean():.2f}")
print(f"  Min: {inun['height_sac'].min():.2f}")
print(f"  Max: {inun['height_sac'].max():.2f}")

## Year-by-year breakdown

In [ ]:
# Add year column
inun['year'] = pd.to_datetime(inun['date']).dt.year

# Group by year
yearly = inun.groupby('year').agg({
    'inundation': 'sum',      # days inundated
    'height_sac': ['mean', 'max'],
    'sac': 'mean'
}).round(1)

yearly.columns = ['Days_Inundated', 'Mean_Height', 'Max_Height', 'Mean_Flow']
yearly.head(20)

In [ ]:
# What year had the most inundation?
max_inun_year = yearly['Days_Inundated'].idxmax()
max_inun_days = yearly['Days_Inundated'].max()
print(f"Year with most inundation: {max_inun_year} ({int(max_inun_days)} days)")

# Least?
min_inun_year = yearly['Days_Inundated'].idxmin()
min_inun_days = yearly['Days_Inundated'].min()
print(f"Year with least inundation: {min_inun_year} ({int(min_inun_days)} days)")

## Quick visualizations

In [ ]:
# Stage height over time
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(inun['date'], inun['height_sac'], linewidth=0.5, alpha=0.7)
ax.axhline(y=33.5, color='r', linestyle='--', label='Pre-2016 threshold (33.5 ft)', alpha=0.7)
ax.axhline(y=32.0, color='orange', linestyle='--', label='Post-2016 threshold (32.0 ft)', alpha=0.7)
ax.set_xlabel('Date')
ax.set_ylabel('Stage Height (feet)')
ax.set_title('Sacramento River Stage Height at Fremont Weir')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Inundation by year
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(yearly.index, yearly['Days_Inundated'], width=0.8, alpha=0.7)
ax.set_xlabel('Year')
ax.set_ylabel('Days Inundated')
ax.set_title('Yolo Bypass Inundation by Year')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Sacramento River flow vs Yolo flow
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(inun['date'], inun['sac'], label='Sacramento River', linewidth=0.5, alpha=0.7)
ax.plot(inun['date'], inun['yolo_dayflow'], label='Yolo Bypass', linewidth=0.5, alpha=0.7)
ax.set_xlabel('Date')
ax.set_ylabel('Flow (cfs)')
ax.set_title('Sacramento River vs Yolo Bypass Flow')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Filter to recent years

In [ ]:
# Just look at 2020-present
recent = inun[inun['date'] >= '2020-01-01']
print(f"Records from 2020-present: {len(recent)}")
print(f"Inundation days in this period: {recent['inundation'].sum()}")
recent.head()

In [ ]:
# Plot recent data
fig, axes = plt.subplots(3, 1, figsize=(14, 8))

# Stage height
axes[0].plot(recent['date'], recent['height_sac'], linewidth=1, color='blue')
axes[0].axhline(y=32.0, color='r', linestyle='--', alpha=0.5)
axes[0].set_ylabel('Stage Height (feet)')
axes[0].set_title('Sacramento River Stage Height (2020-present)')
axes[0].grid(True, alpha=0.3)

# Flow
axes[1].plot(recent['date'], recent['sac'], label='Sacramento', linewidth=1)
axes[1].plot(recent['date'], recent['yolo_dayflow'], label='Yolo', linewidth=1)
axes[1].set_ylabel('Flow (cfs)')
axes[1].set_title('Flow Data (2020-present)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Inundation
axes[2].fill_between(recent['date'], 0, recent['inundation'], alpha=0.5, color='orange', label='Inundated')
axes[2].set_ylabel('Inundation (0/1)')
axes[2].set_xlabel('Date')
axes[2].set_title('Inundation Status (2020-present)')
axes[2].set_ylim(-0.1, 1.1)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Looking at specific events

In [ ]:
# Find the longest inundation event
max_inund_days = inun['inund_days'].max()
max_event_idx = inun[inun['inund_days'] == max_inund_days].index[0]
max_event_start = inun.loc[max_event_idx, 'date']

print(f"Longest inundation event: {int(max_inund_days)} consecutive days")
print(f"Starting around: {max_event_start.date()}")

# Show the event
event = inun[(inun['date'] >= max_event_start - pd.Timedelta(days=5)) & 
             (inun['date'] <= max_event_start + pd.Timedelta(days=int(max_inund_days) + 5))]
print(f"\nEvent details:")
print(event[['date', 'height_sac', 'sac', 'inund_days', 'inundation']].head(20))

## Saving results

In [ ]:
# Save the full results
import os
output_dir = 'data/output'
os.makedirs(output_dir, exist_ok=True)

inun.to_csv(f'{output_dir}/inundation_full.csv', index=False)
print(f"Saved: {output_dir}/inundation_full.csv")

# Save just recent data
recent.to_csv(f'{output_dir}/inundation_2020_present.csv', index=False)
print(f"Saved: {output_dir}/inundation_2020_present.csv")

# Save yearly summary
yearly.to_csv(f'{output_dir}/inundation_by_year.csv')
print(f"Saved: {output_dir}/inundation_by_year.csv")

## Cache management

The package caches data locally for fast repeated access. Let's check it out:

In [ ]:
# See what's cached
cached = show_cache()
for f in cached:
    print(f)

In [ ]:
# Subsequent calls are MUCH faster because they use the cache
import time

# This will be fast (from cache)
start = time.time()
fre_cached = get_fre(use_cache=True)
cached_time = time.time() - start

print(f"Time with cache: {cached_time:.3f} seconds")
print(f"Got {len(fre_cached)} records")

In [ ]:
# Force a fresh download (no cache) - this will be slower
start = time.time()
fre_fresh = get_fre(use_cache=False)
fresh_time = time.time() - start

print(f"Time without cache (fresh download): {fresh_time:.3f} seconds")
print(f"Speedup: {fresh_time/cached_time:.1f}x faster with cache!")

## More exploration ideas

Here are some things you could try:

In [ ]:
# Inundation in spring months (April-May-June)
inun['month'] = pd.to_datetime(inun['date']).dt.month
spring = inun[inun['month'].isin([4, 5, 6])]
spring_inun_pct = 100 * spring['inundation'].mean()
print(f"Inundation rate in spring (Apr-Jun): {spring_inun_pct:.1f}%")

In [ ]:
# What's the relationship between flow and stage height?
import numpy as np

# Scatter plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(inun['sac'], inun['height_sac'], alpha=0.1, s=1)
ax.set_xlabel('Sacramento River Flow (cfs)')
ax.set_ylabel('Stage Height (feet)')
ax.set_title('Relationship between Flow and Stage Height')
ax.grid(True, alpha=0.3)

# Add correlation
corr = inun['sac'].corr(inun['height_sac'])
ax.text(0.05, 0.95, f'Correlation: {corr:.2f}', transform=ax.transAxes, 
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
        verticalalignment='top')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution of inundation days (when inundation happens, how long does it last?)
# Get max inund_days for each event
events = inun[inun['inund_days'] > 0].groupby(
    (inun['inund_days'] == 0).cumsum()
)['inund_days'].max()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(events, bins=50, edgecolor='black', alpha=0.7)
ax.set_xlabel('Duration of Inundation Event (days)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Inundation Event Durations')
ax.grid(True, alpha=0.3, axis='y')
print(f"\nEvent duration statistics:")
print(f"  Mean: {events.mean():.1f} days")
print(f"  Median: {events.median():.1f} days")
print(f"  Max: {events.max():.1f} days")
print(f"  Total events: {len(events)}")
plt.tight_layout()
plt.show()

## That's it!

The inundation package makes it easy to:
- Download real-time data from CDEC and CNRA
- Calculate when the Yolo Bypass is inundated
- Analyze patterns and trends
- Export results for further analysis

Check out the README for more info, or run `help(calc_inundation)` for details on what each function does!